In [4]:
import pandas as pd
import pickle
import os
from string import punctuation
import random

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Variable**

In [5]:
stemmer = SnowballStemmer('english')
lemma = WordNetLemmatizer()
eng_stopwords = stopwords.words('english')

### **Preprocessing**

In [6]:
def alter_tag(tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'

def Preprocess(docx):
    tokens = word_tokenize(docx)
    tokens = [tok.lower() for tok in tokens]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in eng_stopwords]
    tokens = [tok for tok in tokens if tok not in punctuation]

    tokens = [stemmer.stem(tok) for tok in tokens]
    
    tagged = pos_tag(tokens)
    
    tokens = [lemma.lemmatize(tok, alter_tag(tag)) for tok, tag in tagged]

    return tokens


### **Training**

In [7]:
def Training():
    data = pd.read_csv('./Dataset/Tweets.csv')
    X = data['text']
    Y = data['airline_sentiment']

    # Feature Extraction
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocess(text)
        
        feat = {word: True for word in clean}
        feats.append((feat, label))

    random.shuffle(feats)


    # Training
    split = int(0.8 * len(feats))
    train_data = feats[:split]
    evals_data = feats[split:]

    print('Start Training...')
    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print(f'Traning Accuracy: {(acc * 100):.2f}')

    # Info
    print('')
    print('Most Informative Features')
    model.show_most_informative_features(5)

    # Save Model
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')

    return model

def Load():
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        model = Training()
        return model

### **Support Menu**

In [11]:
def Input_Text():
    while True:
        docx = input('Input your Text: ')

        if len(docx.split()) < 5:
            print('The Tweet must be 5 or more words')
        else:
            break
    return docx

def Analyze_Text(model, docx: str):
    if len(docx.split()) < 5:
        print('Please input the Tweet first')
        return
    
    tokens = word_tokenize(docx)
    tokens = [tok.lower() for tok in tokens]
    tokens = [tok for tok in tokens if tok.isalpha()]


    # POS Tag
    tagged = pos_tag(tokens)
    print('The POS Tagged words are:')
    for i, (word, tag) in enumerate(tagged):
        print(f'{i}. {word} : {tag}')
    print('')

    # Syn & Ant
    print('The Synonyms & Antonyms are:')
    for word in tokens:
        print(F'\nWord: {word}')
        synsets = wordnet.synsets(word)
        synonyms = []
        antonyms = []

        for sys in synsets:
            for lemms in sys.lemmas():
                synonyms.append(lemms.name())
                for anton in lemms.antonyms():
                    antonyms.append(anton.name()) 

        print('')
        if len(synonyms) == 0:
            print('No Synonyms')
        else:
            for w in synonyms:
                print(f'(+) {w}')
        
        print()
        if len(antonyms) == 0:
            print('No Antonyms')
        else:
            for w in antonyms:
                print(f'(-) {w}')

    # Predict
    clean = Preprocess(docx)
    feats = {word: True for word in clean}

    category = model.classify(feats)
    print(f'Category: {category}')
    

### **Main**

In [ ]:
def Menu():
    docx = ''
    model = Load()

    while True:
        print('')
        print('1. Write Tweet')
        print('2. Analyze Tweet')
        print('3. End Session')

        cc = input('>> ')
        
        if (cc == '1'):
            docx = Input_Text()
        elif (cc == '2'):
            Analyze_Text(model, docx)
        elif (cc == '3'):
            print('Alright, Thanks for using our App ~~ :)')
            break
        else:
            print('Invalid Input')

In [13]:
Menu()

1. Write Tweet
2. Analyze Tweet
3. End Session
The Tweet must be 5 or more words
The Tweet must be 5 or more words
1. Write Tweet
2. Analyze Tweet
3. End Session
The POS Tagged words are:
0. the : DT
1. tweet : NN
2. must : MD
3. be : VB
4. or : CC
5. more : JJR
6. words : NNS

The Synonyms & Antonyms are:

Word: the

No Synonyms

No Antonyms

Word: tweet

(+) tweet
(+) tweet
(+) twirp
(+) pinch
(+) squeeze
(+) twinge
(+) tweet
(+) nip
(+) twitch

No Antonyms

Word: must

(+) must
(+) must
(+) mustiness
(+) must
(+) moldiness
(+) must

No Antonyms

Word: be

(+) beryllium
(+) Be
(+) glucinium
(+) atomic_number_4
(+) be
(+) be
(+) be
(+) exist
(+) be
(+) be
(+) equal
(+) be
(+) constitute
(+) represent
(+) make_up
(+) comprise
(+) be
(+) be
(+) follow
(+) embody
(+) be
(+) personify
(+) be
(+) be
(+) live
(+) be
(+) cost
(+) be

(-) differ

Word: or

(+) Oregon
(+) Beaver_State
(+) OR
(+) operating_room
(+) OR
(+) operating_theater
(+) operating_theatre
(+) surgery

No Antonyms

Word: m